# Phase 6: Deployment Pipeline
## MAI603 — Machine Learning | Peter Yacoub

**Model:** STGTransformer v2 (GNN + Transformer, d_model=128)  
**Dataset:** PEMS-BAY (325 sensors, SF Bay Area)  
**Stack:** FastAPI + Docker + Streamlit

---

### First-Time Setup (run once)
`Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5 → Cell 6 → Cell 7 → Cell 8 → Cell 9 → Cell 10 → Cell 11`

### After Session Restart (run to restore)
`Cell 1 → Cell 8 (restart API) → Cell 10 (restart Streamlit) → Cell 11 (get URL)`

## Cell 1 — Setup
Mount Drive and initialize all folders and paths.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, pickle
import numpy as np
import torch

DRIVE_FOLDER    = '/content/drive/MyDrive/traffic_project'
DEPLOY_FOLDER   = f'{DRIVE_FOLDER}/deployment'
DASHBOARD_FOLDER = f'{DEPLOY_FOLDER}/dashboard'

os.makedirs(DEPLOY_FOLDER,   exist_ok=True)
os.makedirs(DASHBOARD_FOLDER, exist_ok=True)

print("✅ Deployment folders ready:")
print(f"   {DEPLOY_FOLDER}")
print(f"   {DASHBOARD_FOLDER}")

# Verify model checkpoint exists
ckpt_path = f'{DRIVE_FOLDER}/stgt_v2_best.pt'
if os.path.exists(ckpt_path):
    size_mb = os.path.getsize(ckpt_path) / 1024**2
    print(f"\n✅ Model checkpoint found: {size_mb:.1f} MB")
else:
    print("\n❌ Model checkpoint not found — check Drive")

## Cell 2 — Write model.py
Defines the STGTransformer architecture and `TrafficPredictor` loader class.
Saved to Drive → `deployment/model.py`

In [ ]:
model_py = '''import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pickle
import os

# ── Model Architecture ────────────────────────────────────────
class DiffusionConvLayer(nn.Module):
    def __init__(self, d_model, n_hops=2):
        super().__init__()
        self.fwd_linears = nn.ModuleList([
            nn.Linear(d_model, d_model, bias=False) for _ in range(n_hops)
        ])
        self.bwd_linears = nn.ModuleList([
            nn.Linear(d_model, d_model, bias=False) for _ in range(n_hops)
        ])
        self.out  = nn.Linear(d_model * 2, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, adj_fwd, adj_bwd):
        h_fwd = x
        for layer in self.fwd_linears:
            h_fwd = F.relu(layer(
                torch.einsum("nm, bmd -> bnd", adj_fwd, h_fwd)
            ))
        h_bwd = x
        for layer in self.bwd_linears:
            h_bwd = F.relu(layer(
                torch.einsum("nm, bmd -> bnd", adj_bwd, h_bwd)
            ))
        combined = torch.cat([h_fwd, h_bwd], dim=-1)
        return self.norm(self.out(combined) + x)


class SpatioTemporalTransformer(nn.Module):
    def __init__(self, n_features, d_model, n_heads, n_gnn_layers,
                 n_tf_layers, n_sensors, pred_steps, dropout=0.1):
        super().__init__()
        self.n_sensors  = n_sensors
        self.pred_steps = pred_steps
        self.d_model    = d_model
        INPUT_STEPS     = 12

        self.input_proj = nn.Linear(n_features, d_model)
        self.gnn_layers = nn.ModuleList([
            DiffusionConvLayer(d_model) for _ in range(n_gnn_layers)
        ])
        self.spatial_gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.Sigmoid()
        )
        self.pos_embedding = nn.Parameter(
            torch.randn(1, INPUT_STEPS, d_model) * 0.02
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=n_tf_layers
        )
        self.tf_norm   = nn.LayerNorm(d_model)
        self.pred_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, pred_steps)
        )

    def forward(self, x, adj_fwd, adj_bwd):
        B, T, N, n_feat = x.shape
        x      = self.input_proj(x)
        x_orig = x.clone()
        x_flat = x.reshape(B * T, N, self.d_model)
        for gnn in self.gnn_layers:
            x_flat = gnn(x_flat, adj_fwd, adj_bwd)
        x_spatial = x_flat.reshape(B, T, N, self.d_model)
        gate = self.spatial_gate(torch.cat([x_spatial, x_orig], dim=-1))
        x    = gate * x_spatial + (1 - gate) * x_orig
        x    = x.permute(0, 2, 1, 3).reshape(B * N, T, self.d_model)
        x    = x + self.pos_embedding[:, :T, :]
        x    = self.transformer(x)
        x    = self.tf_norm(x[:, -1, :])
        pred = self.pred_head(x)
        pred = pred.reshape(B, N, self.pred_steps)
        pred = pred.permute(0, 2, 1)
        return pred


# ── Model Loader ──────────────────────────────────────────────
class TrafficPredictor:
    """
    Wraps the trained model with everything needed for inference.
    Loads once at startup, reused for every prediction request.
    """
    def __init__(self, checkpoint_path: str, adj_path: str,
                 scaler_path: str, config_path: str):

        with open(config_path) as f:
            import json
            self.config = json.load(f)

        self.device     = torch.device("cpu")  # CPU for API serving
        self.n_sensors  = self.config["n_sensors"]
        self.n_features = self.config["n_features"]
        self.input_steps = self.config["input_steps"]
        self.pred_steps  = self.config["pred_steps"]
        self.feature_cols = self.config["feature_cols"]

        # Load scaler
        with open(scaler_path, "rb") as f:
            self.scaler = pickle.load(f)
        self.speed_std  = float(self.scaler.scale_[0])
        self.speed_mean = float(self.scaler.mean_[0])

        # Load adjacency matrices
        adj_tensor = torch.load(adj_path, map_location="cpu")
        def norm_adj(a):
            r = a.sum(dim=1, keepdim=True).clamp(min=1e-8)
            return a / r
        self.adj_fwd = norm_adj(adj_tensor)
        self.adj_bwd = norm_adj(adj_tensor.T)

        # Build and load model
        self.model = SpatioTemporalTransformer(
            n_features=self.n_features, d_model=128, n_heads=8,
            n_gnn_layers=2, n_tf_layers=2,
            n_sensors=self.n_sensors, pred_steps=self.pred_steps,
            dropout=0.0
        )
        ckpt = torch.load(checkpoint_path, map_location="cpu")
        self.model.load_state_dict(ckpt["model_state_dict"])
        self.model.eval()
        print(f"✅ Model loaded (epoch {ckpt[\'epoch\']}, val_loss={ckpt[\'val_loss\']:.4f})")

    def predict(self, history: np.ndarray) -> dict:
        """
        history: numpy array of shape (12, 325, 12)
                 — last 12 timesteps, all sensors, all features
        Returns: dict with predictions at 15, 30, 60 min in mph
        """
        x = torch.tensor(history[np.newaxis], dtype=torch.float32)
        with torch.no_grad():
            pred = self.model(x, self.adj_fwd, self.adj_bwd)
        pred_np = pred[0].numpy()   # (12, 325)

        # Convert scaled predictions back to mph
        speed_preds = pred_np * self.speed_std + self.speed_mean

        return {
            "15min": speed_preds[2].tolist(),   # step 3  = 15 min
            "30min": speed_preds[5].tolist(),   # step 6  = 30 min
            "60min": speed_preds[11].tolist(),  # step 12 = 60 min
        }
'''

with open(f'{DEPLOY_FOLDER}/model.py', 'w') as f:
    f.write(model_py)
print("✅ model.py written")

## Cell 3 — Write preprocess.py
Feature engineering pipeline — converts raw sensor readings into model input format.
Saved to Drive → `deployment/preprocess.py`

In [ ]:
preprocess_py = '''import numpy as np
import pickle
import json
from typing import Optional

# Weather condition mapping (same as training)
WEATHER_MAP = {
    "Clear": 0, "Cloudy": 1, "Fog": 2, "Haze": 3,
    "Light Rain": 4, "Light Snow": 5, "Mostly Cloudy": 6,
    "Overcast": 7, "Rain": 8, "Snow": 9, "Unknown": 10
}

def encode_weather(condition: str) -> int:
    """Convert weather string to integer code."""
    return WEATHER_MAP.get(condition, WEATHER_MAP["Unknown"])


def build_feature_vector(
    speed: float,
    hour: int,
    dayofweek: int,
    temperature_f: float,
    humidity_pct: float,
    visibility_mi: float,
    wind_speed_mph: float,
    weather_condition: str,
    acc_count_60min: int,
    acc_max_severity: int,
    acc_mins_since: float,
) -> np.ndarray:
    """
    Build a single feature vector matching the training format.
    Order must match FEATURE_COLS from config:
    [speed, hour, dayofweek, is_weekend, temp, humidity,
     visibility, wind, weather_code, acc_count, acc_severity, acc_mins]
    """
    is_weekend = 1 if dayofweek >= 5 else 0
    weather_code = encode_weather(weather_condition)

    return np.array([
        speed,
        hour,
        dayofweek,
        is_weekend,
        temperature_f,
        humidity_pct,
        visibility_mi,
        wind_speed_mph,
        weather_code,
        acc_count_60min,
        acc_max_severity,
        acc_mins_since,
    ], dtype=np.float32)


def normalize_features(
    feature_vector: np.ndarray,
    scaler
) -> np.ndarray:
    """Apply the same StandardScaler used during training."""
    return (feature_vector - scaler.mean_) / scaler.scale_
'''

with open(f'{DEPLOY_FOLDER}/preprocess.py', 'w') as f:
    f.write(preprocess_py)
print("✅ preprocess.py written")

## Cell 4 — Write main.py (FastAPI)
The REST API with `/predict`, `/health`, and `/sensors` endpoints.
Saved to Drive → `deployment/main.py`

In [ ]:
main_py = '''from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional
import numpy as np
import os
import sys

# Add deployment folder to path
sys.path.append(os.path.dirname(__file__))
from model import TrafficPredictor
from preprocess import build_feature_vector, normalize_features

# ── App Setup ─────────────────────────────────────────────────
app = FastAPI(
    title="Traffic Congestion Predictor",
    description="""
    Real-time traffic speed prediction using a Spatiotemporal
    Transformer + Graph Neural Network model trained on PEMS-BAY
    (325 sensors, San Francisco Bay Area).

    Multi-modal inputs: traffic speed history + weather + accidents.
    Predicts future speeds at 15, 30, and 60 minute horizons.
    """,
    version="1.0.0"
)

# ── Load Model Once at Startup ────────────────────────────────
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
DATA_DIR = os.path.join(BASE_DIR, "model_files")

predictor = TrafficPredictor(
    checkpoint_path = os.path.join(DATA_DIR, "stgt_v2_best.pt"),
    adj_path        = os.path.join(DATA_DIR, "adj_tensor.pt"),
    scaler_path     = os.path.join(DATA_DIR, "scaler.pkl"),
    config_path     = os.path.join(DATA_DIR, "config.json"),
)

# ── Request / Response Models ─────────────────────────────────
class SensorReading(BaseModel):
    sensor_id:         int   = Field(..., description="Sensor index (0-324)")
    speed_mph:         float = Field(..., description="Current speed in mph")
    hour:              int   = Field(..., description="Hour of day (0-23)")
    dayofweek:         int   = Field(..., description="Day of week (0=Mon, 6=Sun)")
    temperature_f:     float = Field(65.0, description="Temperature in Fahrenheit")
    humidity_pct:      float = Field(60.0, description="Humidity percentage")
    visibility_mi:     float = Field(10.0, description="Visibility in miles")
    wind_speed_mph:    float = Field(5.0,  description="Wind speed in mph")
    weather_condition: str   = Field("Clear", description="Weather condition string")
    acc_count_60min:   int   = Field(0,   description="Nearby accidents in past 60 min")
    acc_max_severity:  int   = Field(0,   description="Max severity (0-4)")
    acc_mins_since:    float = Field(60.0,description="Minutes since last accident")


class PredictionRequest(BaseModel):
    """Send the last 12 timesteps (1 hour) of sensor readings."""
    history: List[SensorReading] = Field(
        ..., min_items=12, max_items=12,
        description="Exactly 12 consecutive 5-min readings per sensor"
    )


class SpeedPrediction(BaseModel):
    sensor_id:  int
    speed_15min: float
    speed_30min: float
    speed_60min: float
    congestion_level_15min: str
    congestion_level_30min: str
    congestion_level_60min: str


class PredictionResponse(BaseModel):
    status:      str
    model:       str
    predictions: List[SpeedPrediction]
    summary:     dict


# ── Helper ────────────────────────────────────────────────────
def speed_to_congestion(speed_mph: float) -> str:
    if speed_mph >= 60:  return "Low"
    if speed_mph >= 40:  return "Medium"
    return "High"


# ── Endpoints ─────────────────────────────────────────────────
@app.get("/")
def root():
    return {
        "message": "Traffic Congestion Predictor API",
        "model":   "STGTransformer (GNN + Transformer, d_model=128)",
        "dataset": "PEMS-BAY (325 sensors, SF Bay Area)",
        "horizons": ["15 min", "30 min", "60 min"],
        "docs":    "/docs"
    }


@app.get("/health")
def health():
    return {"status": "healthy", "model_loaded": True}


@app.post("/predict", response_model=PredictionResponse)
def predict(request: PredictionRequest):
    """
    Predict traffic speed for the next 15, 30, and 60 minutes.

    Send 12 consecutive sensor readings (1 hour of history).
    Each reading must include speed, time, weather, and accident context.
    Returns predicted speeds and congestion levels per sensor.
    """
    try:
        n_sensors  = predictor.n_sensors
        n_features = predictor.n_features
        input_steps = predictor.input_steps

        # Build history array: (12, 325, 12)
        history = np.zeros((input_steps, n_sensors, n_features), dtype=np.float32)

        for t, reading in enumerate(request.history):
            sid = reading.sensor_id
            if not (0 <= sid < n_sensors):
                raise HTTPException(
                    status_code=400,
                    detail=f"sensor_id {sid} out of range (0-{n_sensors-1})"
                )
            fv = build_feature_vector(
                speed          = reading.speed_mph,
                hour           = reading.hour,
                dayofweek      = reading.dayofweek,
                temperature_f  = reading.temperature_f,
                humidity_pct   = reading.humidity_pct,
                visibility_mi  = reading.visibility_mi,
                wind_speed_mph = reading.wind_speed_mph,
                weather_condition = reading.weather_condition,
                acc_count_60min   = reading.acc_count_60min,
                acc_max_severity  = reading.acc_max_severity,
                acc_mins_since    = reading.acc_mins_since,
            )
            history[t, sid, :] = normalize_features(fv, predictor.scaler)

        # Run model
        preds = predictor.predict(history)

        # Build response
        sensor_ids = list(set(r.sensor_id for r in request.history))
        response_preds = []
        for sid in sensor_ids:
            s15 = round(preds["15min"][sid], 2)
            s30 = round(preds["30min"][sid], 2)
            s60 = round(preds["60min"][sid], 2)
            response_preds.append(SpeedPrediction(
                sensor_id            = sid,
                speed_15min          = s15,
                speed_30min          = s30,
                speed_60min          = s60,
                congestion_level_15min = speed_to_congestion(s15),
                congestion_level_30min = speed_to_congestion(s30),
                congestion_level_60min = speed_to_congestion(s60),
            ))

        avg_15 = round(np.mean(preds["15min"]), 2)
        avg_30 = round(np.mean(preds["30min"]), 2)
        avg_60 = round(np.mean(preds["60min"]), 2)

        return PredictionResponse(
            status      = "success",
            model       = "STGTransformer v2 (d_model=128)",
            predictions = response_preds,
            summary = {
                "avg_speed_15min_mph": avg_15,
                "avg_speed_30min_mph": avg_30,
                "avg_speed_60min_mph": avg_60,
                "avg_congestion_15min": speed_to_congestion(avg_15),
                "avg_congestion_30min": speed_to_congestion(avg_30),
                "avg_congestion_60min": speed_to_congestion(avg_60),
            }
        )

    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.get("/sensors/count")
def sensor_count():
    return {"n_sensors": predictor.n_sensors}


@app.get("/sensors/{sensor_id}/info")
def sensor_info(sensor_id: int):
    if not (0 <= sensor_id < predictor.n_sensors):
        raise HTTPException(status_code=404, detail="Sensor not found")
    return {
        "sensor_id": sensor_id,
        "location":  "SF Bay Area highway network",
        "sampling":  "Every 5 minutes",
        "features":  predictor.feature_cols,
    }
'''

with open(f'{DEPLOY_FOLDER}/main.py', 'w') as f:
    f.write(main_py)
print("✅ main.py written")

## Cell 5 — Write requirements.txt and Dockerfile
All Python dependencies and Docker container definition.
Saved to Drive → `deployment/requirements.txt` and `deployment/Dockerfile`

In [ ]:
requirements = """fastapi==0.111.0
uvicorn==0.29.0
pydantic==2.7.0
torch==2.3.0
numpy==1.26.4
scikit-learn==1.4.2
streamlit==1.35.0
plotly==5.22.0
requests==2.31.0
pandas==2.2.2
"""

with open(f'{DEPLOY_FOLDER}/requirements.txt', 'w') as f:
    f.write(requirements)
print("✅ requirements.txt written")

dockerfile = """FROM python:3.10-slim

WORKDIR /app

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application files
COPY main.py .
COPY model.py .
COPY preprocess.py .
COPY model_files/ ./model_files/

# Expose FastAPI port
EXPOSE 8000

# Start the API
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

with open(f'{DEPLOY_FOLDER}/Dockerfile', 'w') as f:
    f.write(dockerfile)
print("✅ Dockerfile written")

## Cell 6 — Write Streamlit Dashboard
Interactive web dashboard with sensor map, speed forecast chart, and model performance table.
Saved to Drive → `deployment/dashboard/app.py`

In [ ]:
dashboard_py = '''import streamlit as st
import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import json
import random
from datetime import datetime

# ── Page Config ───────────────────────────────────────────────
st.set_page_config(
    page_title="Traffic Congestion Predictor",
    page_icon="🚦",
    layout="wide"
)

API_URL = "http://localhost:8000"

# ── Sensor GPS coordinates (from PEMS-BAY metadata) ──────────
# A sample of sensor locations for the map
SENSOR_LOCATIONS = {
    0:   (37.2517, -121.9581),
    10:  (37.2507, -121.9125),
    20:  (37.2655, -121.9815),
    30:  (37.3031, -122.0345),
    40:  (37.3516, -122.0603),
    50:  (37.3837, -122.0681),
    60:  (37.4037, -122.0698),
    70:  (37.2627, -121.8585),
    80:  (37.2918, -121.8718),
    90:  (37.3204, -121.8903),
    100: (37.3626, -121.9175),
    110: (37.3746, -121.9311),
    120: (37.3912, -121.9955),
    130: (37.4021, -122.0416),
    140: (37.3297, -121.8421),
}

# ── Helper Functions ──────────────────────────────────────────
def speed_to_color(speed):
    if speed >= 60: return "#2ECC71"   # green  = low congestion
    if speed >= 40: return "#F39C12"   # orange = medium
    return "#E74C3C"                    # red    = high

def speed_to_congestion(speed):
    if speed >= 60: return "🟢 Low"
    if speed >= 40: return "🟡 Medium"
    return "🔴 High"

def make_sample_request(sensor_id, hour, dayofweek, weather):
    """Generate a realistic sample prediction request."""
    base_speed = 65 if hour not in range(7, 10) and hour not in range(16, 19) else 45
    history = []
    for t in range(12):
        history.append({
            "sensor_id":         sensor_id,
            "speed_mph":         base_speed + random.uniform(-5, 5),
            "hour":              hour,
            "dayofweek":         dayofweek,
            "temperature_f":     68.0,
            "humidity_pct":      65.0,
            "visibility_mi":     10.0,
            "wind_speed_mph":    5.0,
            "weather_condition": weather,
            "acc_count_60min":   0,
            "acc_max_severity":  0,
            "acc_mins_since":    60.0,
        })
    return {"history": history}


# ── Sidebar ───────────────────────────────────────────────────
st.sidebar.title("🚦 Traffic Predictor")
st.sidebar.markdown("**STGTransformer v2**")
st.sidebar.markdown("GNN + Transformer | PEMS-BAY")
st.sidebar.divider()

sensor_id  = st.sidebar.selectbox("Sensor ID", list(SENSOR_LOCATIONS.keys()))
hour       = st.sidebar.slider("Hour of Day", 0, 23, 8)
dayofweek  = st.sidebar.selectbox("Day of Week",
    [0,1,2,3,4,5,6],
    format_func=lambda x: ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"][x]
)
weather    = st.sidebar.selectbox("Weather Condition",
    ["Clear","Cloudy","Light Rain","Fog","Overcast"]
)
acc_count  = st.sidebar.slider("Nearby Accidents (past 60 min)", 0, 5, 0)
predict_btn = st.sidebar.button("🔮 Predict", type="primary", use_container_width=True)


# ── Main Content ──────────────────────────────────────────────
st.title("🚦 Real-Time Traffic Congestion Prediction")
st.markdown("**Spatiotemporal Transformer + GNN | SF Bay Area | PEMS-BAY Dataset**")
st.divider()

col1, col2, col3 = st.columns(3)
with col1:
    st.metric("Model", "STGTransformer v2")
with col2:
    st.metric("Sensors", "325")
with col3:
    st.metric("Area", "SF Bay Area")

st.divider()

# ── Prediction ────────────────────────────────────────────────
if predict_btn:
    with st.spinner("Running prediction..."):
        try:
            payload = make_sample_request(sensor_id, hour, dayofweek, weather)
            payload["history"][-1]["acc_count_60min"] = acc_count

            resp = requests.post(f"{API_URL}/predict", json=payload, timeout=30)

            if resp.status_code == 200:
                result = resp.json()
                pred   = result["predictions"][0]
                summ   = result["summary"]

                st.success("✅ Prediction successful")

                # Metrics row
                c1, c2, c3 = st.columns(3)
                with c1:
                    st.metric(
                        "15 min", f"{pred[\'speed_15min\']} mph",
                        delta=f"{pred[\'congestion_level_15min\']}"
                    )
                with c2:
                    st.metric(
                        "30 min", f"{pred[\'speed_30min\']} mph",
                        delta=f"{pred[\'congestion_level_30min\']}"
                    )
                with c3:
                    st.metric(
                        "60 min", f"{pred[\'speed_60min\']} mph",
                        delta=f"{pred[\'congestion_level_60min\']}"
                    )

                # Speed timeline chart
                st.subheader("Speed Forecast Timeline")
                times  = ["Now", "+15 min", "+30 min", "+45 min", "+60 min"]
                speeds = [
                    payload["history"][-1]["speed_mph"],
                    pred["speed_15min"],
                    (pred["speed_15min"] + pred["speed_30min"]) / 2,
                    pred["speed_30min"],
                    pred["speed_60min"],
                ]
                colors = [speed_to_color(s) for s in speeds]
                fig = go.Figure()
                fig.add_trace(go.Scatter(
                    x=times, y=speeds,
                    mode="lines+markers",
                    line=dict(color="#378ADD", width=3),
                    marker=dict(size=12, color=colors),
                    name="Predicted Speed"
                ))
                fig.add_hline(y=60, line_dash="dash",
                              line_color="#2ECC71", annotation_text="Low congestion (60 mph)")
                fig.add_hline(y=40, line_dash="dash",
                              line_color="#E74C3C", annotation_text="High congestion (40 mph)")
                fig.update_layout(
                    yaxis_title="Speed (mph)",
                    yaxis_range=[0, 85],
                    height=350,
                    showlegend=False
                )
                st.plotly_chart(fig, use_container_width=True)

            else:
                st.error(f"API Error {resp.status_code}: {resp.text}")

        except requests.exceptions.ConnectionError:
            st.error("Cannot connect to API. Make sure the FastAPI server is running.")
        except Exception as e:
            st.error(f"Error: {e}")

else:
    st.info("👈 Configure prediction parameters in the sidebar and click **Predict**")


# ── Sensor Map ────────────────────────────────────────────────
st.subheader("📍 PEMS-BAY Sensor Network (Sample)")

map_data = []
for sid, (lat, lon) in SENSOR_LOCATIONS.items():
    map_data.append({
        "sensor_id": sid,
        "lat": lat,
        "lon": lon,
        "speed": random.uniform(40, 75),
    })

df_map = pd.DataFrame(map_data)
df_map["congestion"] = df_map["speed"].apply(
    lambda s: "Low" if s >= 60 else "Medium" if s >= 40 else "High"
)
df_map["color"] = df_map["speed"].apply(speed_to_color)

fig_map = px.scatter_mapbox(
    df_map, lat="lat", lon="lon",
    color="congestion",
    color_discrete_map={"Low":"#2ECC71","Medium":"#F39C12","High":"#E74C3C"},
    size_max=15,
    hover_data={"sensor_id": True, "speed": ":.1f", "lat": False, "lon": False},
    zoom=10,
    mapbox_style="open-street-map",
    title="Sensor Locations (color = congestion level)"
)
fig_map.update_traces(marker=dict(size=12))
fig_map.update_layout(height=450, margin=dict(l=0,r=0,t=30,b=0))
st.plotly_chart(fig_map, use_container_width=True)

# ── Model Info ────────────────────────────────────────────────
with st.expander("📊 Model Performance"):
    perf_data = {
        "Model":    ["Hist. Average","ARIMA","LSTM","STGTransformer (ours)",
                     "DCRNN (2018)","STGCN (2018)"],
        "15 min":   [2.93, 2.98, 1.47, 1.46, 1.38, 1.36],
        "30 min":   [3.11, 2.99, 1.97, 1.91, 1.74, 1.81],
        "60 min":   [3.54, 3.03, 2.61, 2.42, 2.07, 2.49],
    }
    df_perf = pd.DataFrame(perf_data)
    st.dataframe(
        df_perf.style.highlight_min(subset=["15 min","30 min","60 min"],
                                     color="#d4edda"),
        use_container_width=True
    )
    st.caption("MAE in mph. Lower is better. Green = best in column.")
'''

with open(f'{DASHBOARD_FOLDER}/app.py', 'w') as f:
    f.write(dashboard_py)
print("✅ dashboard/app.py written")

## Cell 7 — Copy Model Files + Deploy to Local Disk
Copies checkpoint, adjacency matrix, scaler, and config into the deployment folder.  
Then copies everything to Colab local disk (`/content/deployment`) for faster API execution.

In [ ]:
import shutil

# Create model_files directory inside deployment
MODEL_FILES_DIR = f'{DEPLOY_FOLDER}/model_files'
os.makedirs(MODEL_FILES_DIR, exist_ok=True)

# Files needed for the API to run
files_to_copy = [
    ('stgt_v2_best.pt', 'stgt_v2_best.pt'),
    ('adj_tensor.pt',   'adj_tensor.pt'),
    ('scaler.pkl',      'scaler.pkl'),
    ('config.json',     'config.json'),
]

for src_name, dst_name in files_to_copy:
    src = f'{DRIVE_FOLDER}/{src_name}'
    dst = f'{MODEL_FILES_DIR}/{dst_name}'
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1024**2
        print(f"  ✅ Copied {src_name} ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ Missing: {src_name}")

print(f"\n✅ model_files/ ready")
print(f"\nDeployment folder structure:")
for root, dirs, files in os.walk(DEPLOY_FOLDER):
    level = root.replace(DEPLOY_FOLDER, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        size_mb = os.path.getsize(os.path.join(root, file)) / 1024**2
        print(f"{subindent}{file}  ({size_mb:.1f} MB)")

import shutil, os

# Copy deployment folder to Colab local disk for faster API execution
LOCAL_DEPLOY = '/content/deployment'

if os.path.exists(LOCAL_DEPLOY):
    shutil.rmtree(LOCAL_DEPLOY)

shutil.copytree(DEPLOY_FOLDER, LOCAL_DEPLOY)
print(f"✅ Copied deployment folder to local disk")
print(f"   {LOCAL_DEPLOY}")

# Verify
for root, dirs, files in os.walk(LOCAL_DEPLOY):
    level = root.replace(LOCAL_DEPLOY, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{' ' * 2 * (level+1)}{f}")

## Cell 8 — Start FastAPI Server
Installs dependencies and launches the API on port 8000.  
**Re-run this cell after any session restart to restore the API.**

In [ ]:
# Install required packages for the API server
!pip install -q fastapi uvicorn pydantic torch

import subprocess, threading, time, requests

# Run FastAPI in a background daemon thread so Colab stays responsive
def run_api():
    subprocess.run([
        "uvicorn", "main:app",
        "--host", "0.0.0.0",
        "--port", "8000",
        "--log-level", "warning"
    ], cwd='/content/deployment')   # ← serves from local disk copy

thread = threading.Thread(target=run_api, daemon=True)
thread.start()

# Give the server 10 seconds to initialise before health check
print("Starting FastAPI server...")
time.sleep(10)

# Health check — confirms the server is accepting requests
try:
    resp = requests.get("http://localhost:8000/health")
    if resp.status_code == 200:
        print("✅ FastAPI is running!")
        print(f"   Response: {resp.json()}")
    else:
        print(f"❌ Status {resp.status_code}: {resp.text}")
except Exception as e:
    print(f"❌ Could not connect: {e}")
    print("   Wait 10 more seconds and try /health again")


## Cell 9 — Test the /predict Endpoint
Sends a real prediction request simulating Monday 8 AM rush hour with light rain and 1 nearby accident.  
Expected output: Medium congestion (~50-55 mph) for sensor 0.

In [ ]:
import json
from datetime import datetime

# Build a test request — simulating rush hour conditions
test_payload = {
    "history": [
        {
            "sensor_id":         0,
            "speed_mph":         45.0 + i * 0.5,
            "hour":              8,
            "dayofweek":         0,
            "temperature_f":     58.0,
            "humidity_pct":      75.0,
            "visibility_mi":     8.0,
            "wind_speed_mph":    6.0,
            "weather_condition": "Light Rain",
            "acc_count_60min":   1,
            "acc_max_severity":  2,
            "acc_mins_since":    15.0,
        }
        for i in range(12)   # 12 timesteps = 1 hour of history
    ]
}

resp = requests.post("http://localhost:8000/predict", json=test_payload)
result = resp.json()

print("="*55)
print("TEST: Monday 8 AM, Light Rain, 1 nearby accident")
print("="*55)
print(f"Status: {result['status']}")
print(f"Model:  {result['model']}")
print(f"\nPredicted speeds for Sensor 0:")
pred = result['predictions'][0]
print(f"  15 min: {pred['speed_15min']} mph  → {pred['congestion_level_15min']}")
print(f"  30 min: {pred['speed_30min']} mph  → {pred['congestion_level_30min']}")
print(f"  60 min: {pred['speed_60min']} mph  → {pred['congestion_level_60min']}")

print(f"\nNetwork-wide averages:")
summ = result['summary']
print(f"  15 min avg: {summ['avg_speed_15min_mph']} mph  → {summ['avg_congestion_15min']}")
print(f"  30 min avg: {summ['avg_speed_30min_mph']} mph  → {summ['avg_congestion_30min']}")
print(f"  60 min avg: {summ['avg_speed_60min_mph']} mph  → {summ['avg_congestion_60min']}")

print(f"\n✅ API is working correctly")
print(f"\nRaw JSON response:")
print(json.dumps(result, indent=2)[:500] + "...")

## Cell 10 — Start Streamlit Dashboard
Launches the interactive dashboard on port 8501.  
**Re-run this cell after any session restart to restore the dashboard.**

In [ ]:
import subprocess, threading, time

# Kill existing Streamlit
subprocess.run(['fuser', '-k', '8501/tcp'], capture_output=True)
time.sleep(2)

DASHBOARD_FOLDER = '/content/deployment/dashboard'

def run_streamlit():
    subprocess.run([
        "streamlit", "run",
        f"{DASHBOARD_FOLDER}/app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ])

st_thread = threading.Thread(target=run_streamlit, daemon=True)
st_thread.start()
time.sleep(8)
print("✅ Streamlit restarted")

## Cell 11 — Get Dashboard URL
Gets the public URL for the Streamlit dashboard.  
Open the URL in your browser to view the live dashboard.

In [ ]:
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"Dashboard URL: {url}")

## Cell 12 — Build Docker Image
Builds the Docker container from the Dockerfile.  
This proves the project is production-ready and fully containerized.  
*(Run once — takes 3-5 minutes)*

In [ ]:
# ── Optional: Build Docker Image ────────────────────────────────────────────
# This cell packages the entire deployment (model + API + preprocessing)
# into a self-contained Docker image.
#
# ⚠️  Docker may not be available in all Colab sessions.
#     If it fails, the Dockerfile is still written to Drive and ready
#     for local deployment on any machine with Docker installed.
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, os

LOCAL_DEPLOY = '/content/deployment'

print("Building Docker image...")
print("(This takes 3-5 minutes)\n")

result = subprocess.run(
    ["docker", "build", "-t", "traffic-predictor:v1", "."],
    cwd=LOCAL_DEPLOY,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✅ Docker image built successfully!")
    print("\nImage details:")
    subprocess.run(["docker", "images", "traffic-predictor"], check=True)
else:
    print("❌ Docker build failed:")
    print(result.stderr[-2000:])
    print("\nNote: Docker may not be available in this Colab session.")
    print("The Dockerfile is written and ready for local deployment.")


## Cell 13 — Deployment Summary
Prints a complete summary of all deployed components and file locations.

In [ ]:
# ── Deployment Summary ───────────────────────────────────────────────────────
# Checks the status of all deployed components and prints a full summary:
#   - FastAPI server health
#   - Deployment files on Drive (with sizes)
#   - Available API endpoints
#   - Model performance comparison table
# ─────────────────────────────────────────────────────────────────────────────

import os, json

DRIVE_FOLDER  = '/content/drive/MyDrive/traffic_project'
LOCAL_DEPLOY  = '/content/deployment'

print("=" * 65)
print("PHASE 6 — DEPLOYMENT PIPELINE SUMMARY")
print("=" * 65)

print("\n── API Status ──")
try:
    import requests
    resp = requests.get("http://localhost:8000/health", timeout=3)
    print(f"  FastAPI:   ✅ Running  →  {resp.json()}")
except:
    print(f"  FastAPI:   ❌ Not running  (run Cell 8 to start)")

print("\n── Deployment Files (Drive) ──")
deploy_files = {
    'main.py':           'FastAPI application',
    'model.py':          'Model architecture + loader',
    'preprocess.py':     'Feature engineering pipeline',
    'requirements.txt':  'Python dependencies',
    'Dockerfile':        'Container definition',
    'dashboard/app.py':  'Streamlit dashboard',
    'model_files/stgt_v2_best.pt': 'Trained model checkpoint',
    'model_files/adj_tensor.pt':   'Road network graph',
    'model_files/scaler.pkl':      'Normalization scaler',
    'model_files/config.json':     'Model configuration',
}

deploy_path = f'{DRIVE_FOLDER}/deployment'
for fname, desc in deploy_files.items():
    path   = f'{deploy_path}/{fname}'
    exists = '✅' if os.path.exists(path) else '❌'
    size   = f"({os.path.getsize(path)/1024**2:.1f} MB)" if os.path.exists(path) else ''
    print(f"  {exists} {fname:<40} {size}")

print("\n── API Endpoints ──")
endpoints = [
    ('GET',  '/',                  'API info and description'),
    ('GET',  '/health',            'Health check'),
    ('POST', '/predict',           'Traffic speed prediction'),
    ('GET',  '/sensors/count',     'Number of sensors'),
    ('GET',  '/sensors/{id}/info', 'Sensor details'),
    ('GET',  '/docs',              'Interactive API documentation'),
]
for method, path, desc in endpoints:
    print(f"  {method:<5} http://localhost:8000{path:<25} — {desc}")

print("\n── Model Performance ──")
print(f"  {'Model':<28} {'15 min':>8} {'30 min':>8} {'60 min':>8}")
print(f"  {'-'*55}")
results = [
    ('STGTransformer v2 (ours)',  1.46, 1.91, 2.42),
    ('DCRNN (published 2018)',    1.38, 1.74, 2.07),
    ('STGCN (published 2018)',    1.36, 1.81, 2.49),
    ('LSTM (baseline)',           1.47, 1.97, 2.61),
]
for name, v15, v30, v60 in results:
    marker = ' ← OURS' if 'ours' in name.lower() else ''
    print(f"  {name:<28} {v15:>7.2f}  {v30:>7.2f}  {v60:>7.2f} mph{marker}")

print("\n── Project Complete ──")
print("  ✅ Data pipeline (PEMS-BAY + US-Accidents, 16.9M rows)")
print("  ✅ Model training (STGTransformer v2, epoch 50, val=0.0776)")
print("  ✅ Ablation study (multi-modal features verified)")
print("  ✅ SHAP explainability (feature importance charts)")
print("  ✅ FastAPI endpoint (/predict working)")
print("  ✅ Docker container (Dockerfile ready)")
print("  ✅ Streamlit dashboard (live on port 8501)")
print("=" * 65)
